# SimPO Preference Alignment for Socratic Style

This notebook applies **SimPO** (Simple Preference Optimization) to align the SFT-trained
Qwen3 model toward Socratic tutoring behavior using preference pairs.

**Training pipeline stage:** 2 of 4 (SFT -> **SimPO** -> GRPO -> STaR)

SimPO is a reference-model-free variant of DPO that uses the average log probability
of the completion as the implicit reward, eliminating the need for a reference model.

**Preference criteria:**
- Chosen: Socratic responses (guiding questions, scaffolding, encouraging reasoning)
- Rejected: Direct answers, overly verbose explanations, non-pedagogical responses

**Domains:** Mathematics, Physics, Chemistry, Biology, Computer Science

In [ ]:
# Install dependencies
!pip install -q unsloth trl peft transformers datasets
!pip install -q accelerate bitsandbytes sentencepiece protobuf

In [ ]:
# ============================================================
# Configuration
# ============================================================

# Model size selection: "4b" or "1.7b"
MODEL_SIZE = "4b"  # Change to "1.7b" for the smaller model

# Resolve model and checkpoint paths based on size
MODEL_MAP = {
    "4b": {
        "base_model": "unsloth/Qwen3-4B",
        "sft_checkpoint": "/content/drive/MyDrive/MITS/checkpoints/sft_qwen3_4b/final_adapter",
        "output_dir": "/content/drive/MyDrive/MITS/checkpoints/simpo_qwen3_4b",
    },
    "1.7b": {
        "base_model": "unsloth/Qwen3-1.7B",
        "sft_checkpoint": "/content/drive/MyDrive/MITS/checkpoints/sft_qwen3_1.7b/final_adapter",
        "output_dir": "/content/drive/MyDrive/MITS/checkpoints/simpo_qwen3_1.7b",
    },
}

assert MODEL_SIZE in MODEL_MAP, f"MODEL_SIZE must be '4b' or '1.7b', got '{MODEL_SIZE}'"
config = MODEL_MAP[MODEL_SIZE]

BASE_MODEL = config["base_model"]
SFT_CHECKPOINT = config["sft_checkpoint"]
OUTPUT_DIR = config["output_dir"]

# SimPO hyperparameters
LEARNING_RATE = 5e-7
BETA = 2.0   # SimPO beta (controls preference strength)
GAMMA = 1.0  # SimPO gamma (reward margin)
EPOCHS = 1
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
MAX_SEQ = 2048
MAX_PROMPT_LENGTH = 1024

# Paths
PREFERENCE_DATA_PATH = "training/data/preference_pairs.jsonl"
VALIDATION_SPLIT = 0.1

# Domains
DOMAINS = ["math", "physics", "chemistry", "biology", "cs"]

print(f"Model size: {MODEL_SIZE}")
print(f"Base model: {BASE_MODEL}")
print(f"SFT checkpoint: {SFT_CHECKPOINT}")
print(f"SimPO beta: {BETA}, gamma: {GAMMA}")
print(f"LR: {LEARNING_RATE}, Epochs: {EPOCHS}")
print(f"Preference data: {PREFERENCE_DATA_PATH}")

In [ ]:
# ============================================================
# Mount Google Drive (optional) and load preference pairs from HuggingFace
# ============================================================
import json
import os
import random
from collections import Counter

# Try mounting Drive; fall back to local
DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = True
    print("Google Drive mounted successfully")
except Exception as e:
    print(f"Drive mount failed ({e}), using local storage")
    OUTPUT_DIR = f"/content/checkpoints/simpo_qwen3_{MODEL_SIZE}"
    SFT_CHECKPOINT = f"/content/checkpoints/sft_qwen3_{MODEL_SIZE}/final_adapter"

# Load from HuggingFace Hub
from datasets import load_dataset

print("Loading preference pairs from Siesher/mits-stem-training-data...")
hf_ds = load_dataset("Siesher/mits-stem-training-data", "preference")

train_records = [dict(r) for r in hf_ds["train"]]
val_records = [dict(r) for r in hf_ds["test"]]

print(f"Train: {len(train_records)}, Validation: {len(val_records)}")

# Domain distribution
domain_counts = Counter(r.get("domain", "unknown") for r in train_records)
for domain, count in sorted(domain_counts.items()):
    print(f"  {domain}: {count} pairs")

# Verify SFT checkpoint exists
assert os.path.exists(SFT_CHECKPOINT), f"SFT checkpoint not found at {SFT_CHECKPOINT}"
print(f"SFT checkpoint verified: {SFT_CHECKPOINT}")

In [ ]:
# ============================================================
# Load SFT model + adapter
# ============================================================
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    dtype=None,
)

# Load SFT adapter weights
from peft import PeftModel
model = PeftModel.from_pretrained(model, SFT_CHECKPOINT)
model = model.merge_and_unload()  # Merge SFT weights for SimPO training

print(f"Loaded base model: {BASE_MODEL}")
print(f"Merged SFT adapter from: {SFT_CHECKPOINT}")

# Apply fresh LoRA for SimPO training
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

In [ ]:
# ============================================================
# CPOTrainer with loss_type="simpo" and domain-balanced batches
# ============================================================
from trl import CPOConfig, CPOTrainer
from datasets import Dataset
from collections import defaultdict


def format_preference_record(record):
    """Convert a preference record to the format expected by CPOTrainer."""
    prompt_messages = record.get("prompt", [])
    if isinstance(prompt_messages, str):
        prompt = prompt_messages
    else:
        prompt = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    return {
        "prompt": prompt,
        "chosen": record["chosen"],
        "rejected": record["rejected"],
        "domain": record.get("domain", "unknown"),
    }


def create_domain_balanced_dataset(records):
    """Create a domain-balanced dataset by oversampling minority domains."""
    domain_records = defaultdict(list)
    for r in records:
        domain_records[r.get("domain", "unknown")].append(r)

    # Find max domain size
    max_size = max(len(v) for v in domain_records.values())

    # Oversample to balance
    balanced = []
    for domain in DOMAINS:
        domain_data = domain_records.get(domain, [])
        if not domain_data:
            continue
        # Repeat to match max size
        multiplied = domain_data * (max_size // len(domain_data) + 1)
        balanced.extend(multiplied[:max_size])

    random.shuffle(balanced)
    return balanced


# Create balanced training set
balanced_train = create_domain_balanced_dataset(train_records)
print(f"Balanced training set: {len(balanced_train)} pairs")

# Format datasets
train_formatted = [format_preference_record(r) for r in balanced_train]
val_formatted = [format_preference_record(r) for r in val_records]

train_ds = Dataset.from_list(train_formatted)
val_ds = Dataset.from_list(val_formatted)

print(f"Train dataset: {len(train_ds)} examples")
print(f"Val dataset: {len(val_ds)} examples")

# CPOTrainer with SimPO loss
cpo_config = CPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    loss_type="simpo",
    cpo_alpha=0.0,  # Pure SimPO (no NLL component)
    beta=BETA,
    simpo_gamma=GAMMA,
    max_length=MAX_SEQ,
    max_prompt_length=MAX_PROMPT_LENGTH,
    optim="adamw_8bit",
    seed=42,
    report_to="none",
    remove_unused_columns=False,
)

trainer = CPOTrainer(
    model=model,
    args=cpo_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)

print("CPOTrainer configured with SimPO loss")

In [ ]:
# ============================================================
# Train 1 epoch
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Starting SimPO training...")
print(f"  Beta: {BETA}, Gamma: {GAMMA}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")

result = trainer.train()

print(f"\nTraining complete!")
print(f"  Train loss: {result.training_loss:.4f}")
print(f"  Train runtime: {result.metrics.get('train_runtime', 0):.0f}s")
print(f"  Train samples/sec: {result.metrics.get('train_samples_per_second', 0):.1f}")

In [ ]:
# ============================================================
# Evaluate preference accuracy on held-out pairs per domain
# ============================================================
import math

print("Evaluating preference accuracy on held-out pairs...")

# Overall eval
eval_results = trainer.evaluate()
print(f"\nOverall eval metrics:")
for k, v in eval_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

# Per-domain preference accuracy
FastLanguageModel.for_inference(model)

domain_accuracy = defaultdict(lambda: {"correct": 0, "total": 0})

for record in val_records:
    domain = record.get("domain", "unknown")
    formatted = format_preference_record(record)
    prompt = formatted["prompt"]

    # Compute log probabilities for chosen and rejected
    chosen_text = prompt + formatted["chosen"]
    rejected_text = prompt + formatted["rejected"]

    chosen_ids = tokenizer(chosen_text, return_tensors="pt", truncation=True, max_length=MAX_SEQ).to(model.device)
    rejected_ids = tokenizer(rejected_text, return_tensors="pt", truncation=True, max_length=MAX_SEQ).to(model.device)

    with torch.no_grad():
        chosen_logits = model(**chosen_ids).logits
        rejected_logits = model(**rejected_ids).logits

    # Average log probability (SimPO reward)
    prompt_len = len(tokenizer(prompt, truncation=True, max_length=MAX_SEQ)["input_ids"])

    chosen_log_probs = torch.nn.functional.log_softmax(chosen_logits[:, prompt_len-1:-1, :], dim=-1)
    chosen_token_ids = chosen_ids["input_ids"][:, prompt_len:]
    chosen_reward = chosen_log_probs.gather(-1, chosen_token_ids.unsqueeze(-1)).squeeze(-1).mean().item()

    rejected_log_probs = torch.nn.functional.log_softmax(rejected_logits[:, prompt_len-1:-1, :], dim=-1)
    rejected_token_ids = rejected_ids["input_ids"][:, prompt_len:]
    rejected_reward = rejected_log_probs.gather(-1, rejected_token_ids.unsqueeze(-1)).squeeze(-1).mean().item()

    domain_accuracy[domain]["total"] += 1
    if chosen_reward > rejected_reward:
        domain_accuracy[domain]["correct"] += 1

print("\nPer-domain preference accuracy:")
total_correct = 0
total_total = 0
for domain in DOMAINS:
    m = domain_accuracy[domain]
    if m["total"] > 0:
        pct = 100 * m["correct"] / m["total"]
        print(f"  {domain}: {pct:.1f}% ({m['correct']}/{m['total']})")
        total_correct += m["correct"]
        total_total += m["total"]

if total_total > 0:
    print(f"  Overall: {100 * total_correct / total_total:.1f}% ({total_correct}/{total_total})")

# Save evaluation metrics
metrics_path = os.path.join(OUTPUT_DIR, "simpo_eval_metrics.json")
with open(metrics_path, "w") as f:
    json.dump({
        "eval_results": eval_results,
        "domain_accuracy": dict(domain_accuracy),
        "model_size": MODEL_SIZE,
        "beta": BETA,
        "gamma": GAMMA,
    }, f, indent=2, default=str)
print(f"Metrics saved to {metrics_path}")

In [ ]:
# ============================================================
# Save updated adapter
# ============================================================

final_adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)
print(f"SimPO adapter saved to {final_adapter_path}")

# Save training config
config_to_save = {
    "stage": "simpo",
    "model_size": MODEL_SIZE,
    "base_model": BASE_MODEL,
    "sft_checkpoint": SFT_CHECKPOINT,
    "loss_type": "simpo",
    "beta": BETA,
    "gamma": GAMMA,
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "max_seq_length": MAX_SEQ,
    "train_pairs": len(train_records),
    "val_pairs": len(val_records),
}
config_path = os.path.join(OUTPUT_DIR, "training_config.json")
with open(config_path, "w") as f:
    json.dump(config_to_save, f, indent=2)
print(f"Config saved to {config_path}")

print("\nDone! SimPO-aligned adapter is ready for GRPO reinforcement (next stage).")